# UC-VOLUMES-1 — CityJSON to Globe: ingest Den Haag LoD2 buildings

**Persona:** Urban Data Analyst

**Goal:** Apply the `geovolumes_demo` preset to seed a Den Haag (The Hague)
CityJSON LoD2 dataset, poll the ingestion until the 3D container appears with a
non-zero extent, walk the GeoVolumes Core collection API (listing, single
container, 3D tile metadata, spatial bbox query), stream the first lines of the
CityJSONSeq endpoint, and finish with a deep link to the globe browser.

**Prerequisites:**
- GeoID platform running locally at `DYNASTORE_BASE_URL` (default `http://localhost:8080`)
- `volumes` extension enabled in the active SCOPE
- Write-scoped JWT in env var `DYNASTORE_TOKEN` (or IDP client credentials set)
- Internet access for the first run (preset downloads `DenHaag_01.city.json` ~28 MB)

**References:**
- OGC API — 3D GeoVolumes 1.0 working draft
- Preset: `geovolumes_demo` (catalog `demo-3d`, collection `denhaag`)

In [ ]:
import json
import os
import time

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision token via IDP client_credentials when not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass

TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_WRITE_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)

CATALOG_ID = "demo-3d"
COLLECTION_ID = "denhaag"

headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=300.0)

print(f"Platform: {BASE_URL}")
print(f"Catalog : {CATALOG_ID}  Collection: {COLLECTION_ID}")

## Step 1 — Apply the `geovolumes_demo` preset

`POST /configs/presets/{preset_id}?force=true` with a JSON params body seeds the
`demo-3d` catalog and triggers asynchronous CityJSON ingestion. The preset is
idempotent: re-running it skips ingestion when the collection already contains items.

The `force=true` flag reapplies even when the preset sentinel says it ran before.

In [ ]:
preset_params = {"collection_id": COLLECTION_ID}

r = client.post(
    f"/configs/presets/geovolumes_demo",
    params={"force": "true"},
    content=json.dumps(preset_params),
)
print(r.status_code, r.text[:600])

if r.status_code not in (200, 201, 202):
    print(f"\nPreset apply returned {r.status_code} — volumes extension may not be active.")
    print("Ensure the 'volumes' SCOPE is enabled and restart the platform.")
else:
    print("\nPreset accepted. Ingestion is running asynchronously.")

## Step 2 — Poll the GeoVolumes collection list until the container appears

`GET /volumes/catalogs/{catalog_id}/collections` returns a list of collections
flagged as 3D containers (extras carry `cityjson:version` or `geovolumes:enabled`).
We poll until the `denhaag` container appears with a non-zero bounding extent.
Ingestion can take 30–120 s depending on machine speed and network.

In [ ]:
MAX_POLLS = 30
POLL_INTERVAL_S = 10
_collection_ready = False

for attempt in range(1, MAX_POLLS + 1):
    r = client.get(f"/volumes/catalogs/{CATALOG_ID}/collections")
    if r.status_code != 200:
        print(f"[{attempt:2d}] GET collections returned {r.status_code} — retrying")
        time.sleep(POLL_INTERVAL_S)
        continue

    colls = r.json().get("collections", [])
    match = next((c for c in colls if c.get("id") == COLLECTION_ID), None)

    if match:
        extent = match.get("contentExtent", {}).get("bbox", [])
        item_count = match.get("itemCount", 0)
        print(f"[{attempt:2d}] Found '{COLLECTION_ID}': extent={extent}  items={item_count}")
        if extent and item_count and item_count > 0:
            _collection_ready = True
            print("\nCollection ready with non-zero extent.")
            break
        print(f"     extent or item count not yet populated — waiting {POLL_INTERVAL_S}s...")
    else:
        print(f"[{attempt:2d}] Collection not yet visible — waiting {POLL_INTERVAL_S}s...")

    time.sleep(POLL_INTERVAL_S)

if not _collection_ready:
    print("\nTimeout: collection did not reach ready state within the poll window.")
    print("If ingestion is still running, re-run this cell after a short wait.")

## Step 3 — Inspect the single 3D container

`GET /volumes/catalogs/{catalog_id}/collections/{collection_id}` returns a
`ThreeDContainer` with content links for 3D Tiles, CityJSONSeq, and metadata.

In [ ]:
r = client.get(f"/volumes/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
print(r.status_code)

if r.status_code == 200:
    container = r.json()
    print(f"id          : {container.get('id')}")
    print(f"title       : {container.get('title')}")
    print(f"extent.bbox : {container.get('contentExtent', {}).get('bbox')}")
    print(f"itemCount   : {container.get('itemCount')}")
    print("\nContent links:")
    for lnk in container.get("links", []):
        print(f"  rel={lnk.get('rel'):20s}  type={lnk.get('type', '')}  href={lnk.get('href', '')[:80]}")
elif r.status_code == 404:
    print("SKIP: collection not found — run Step 1 + Step 2 first.")
else:
    print(r.text[:400])

## Step 4 — Fetch 3D tile metadata

`GET /volumes/catalogs/{catalog_id}/collections/{collection_id}/3dtiles/metadata`
returns service metadata including tileset config and content links.

In [ ]:
r = client.get(
    f"/volumes/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/3dtiles/metadata"
)
print(r.status_code)
if r.status_code == 200:
    meta = r.json()
    print(f"title            : {meta.get('title')}")
    cfg = meta.get("config", {})
    print(f"max_features/tile: {cfg.get('max_features_per_tile')}")
    print(f"max_tree_depth   : {cfg.get('max_tree_depth')}")
    print(f"formats          : {cfg.get('supported_formats')}")
    for lnk in meta.get("links", []):
        print(f"  {lnk.get('rel'):8s} -> {lnk.get('href', '')[:80]}")
elif r.status_code == 404:
    print("SKIP: collection not found — run steps 1-2 first.")
else:
    print(r.text[:300])

## Step 5 — Spatial bbox filter (hit / miss)

`GET /volumes/catalogs/{catalog_id}/collections?bbox=minLon,minLat,maxLon,maxLat`
filters containers by spatial extent. Den Haag is around lon 4.3, lat 52.07.

We test two bboxes: one that overlaps the collection, one that does not.

In [ ]:
# Bbox that overlaps Den Haag (lon 4.2-4.4, lat 52.0-52.1)
bbox_hit  = "4.2,52.0,4.4,52.1"
# Bbox far away (Gulf of Guinea)
bbox_miss = "-1.0,-1.0,1.0,1.0"

for label, bbox in (("HIT", bbox_hit), ("MISS", bbox_miss)):
    r = client.get(
        f"/volumes/catalogs/{CATALOG_ID}/collections",
        params={"bbox": bbox},
    )
    if r.status_code == 200:
        colls = r.json().get("collections", [])
        ids = [c.get("id") for c in colls]
        print(f"[{label}] bbox={bbox} -> {len(colls)} collection(s): {ids}")
    else:
        print(f"[{label}] status={r.status_code}: {r.text[:200]}")

## Step 6 — Stream first 3 lines of CityJSONSeq

`GET /volumes/catalogs/{catalog_id}/collections/{collection_id}/cityjsonseq`
returns newline-delimited JSON: line 1 is the CityJSONSeq header, subsequent
lines are CityJSONFeature objects. Media type: `application/city+json`.

We stream with `limit=3` to keep the response small.

In [ ]:
with client.stream(
    "GET",
    f"/volumes/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/cityjsonseq",
    params={"limit": 3},
) as resp:
    print(f"status       : {resp.status_code}")
    print(f"content-type : {resp.headers.get('content-type', 'n/a')}")
    if resp.status_code == 200:
        for i, line in enumerate(resp.iter_lines()):
            if i >= 3:
                break
            obj = json.loads(line)
            if i == 0:
                print(f"\nHeader: type={obj.get('type')}  version={obj.get('version')}")
                trs = obj.get("transform", {})
                print(f"  transform.scale    : {trs.get('scale')}")
                print(f"  transform.translate: {trs.get('translate')}")
            else:
                print(f"Feature {i}: id={obj.get('id')}  type={obj.get('type')}")
    elif resp.status_code == 404:
        print("SKIP: collection not found — run steps 1-2 first.")
    else:
        print(resp.text[:300])

## Step 7 — Tileset JSON (OGC 3D Tiles index)

`GET /volumes/catalogs/{catalog_id}/collections/{collection_id}/3dtiles/tileset.json`
returns the BSP-partitioned tileset index that Cesium / MapLibre 3D can consume.

In [ ]:
r = client.get(
    f"/volumes/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/3dtiles/tileset.json"
)
print(f"status: {r.status_code}")
if r.status_code == 200:
    tileset = r.json()
    print(f"asset.version   : {tileset.get('asset', {}).get('version')}")
    root = tileset.get("root", {})
    print(f"root.geometricError: {root.get('geometricError')}")
    bv = root.get("boundingVolume", {})
    print(f"root.boundingVolume: {bv}")
    children = root.get("children", [])
    print(f"root.children count: {len(children)}")
elif r.status_code == 404:
    print("SKIP: collection not found — run steps 1-2 first.")
else:
    print(r.text[:300])

## Result — Open the globe browser

The link below opens the 3D GeoVolumes globe browser for interactive exploration
of the Den Haag buildings.

In [ ]:
client.close()
print(f"Open the globe browser: {BASE_URL}/web/pages/volumes_browser?language=en")
print(f"Direct collection:      {BASE_URL}/volumes/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")